In [1]:
import logging
import platform
import random
import subprocess
import time
from os import getenv

import fs_structs
from dotenv import load_dotenv

from distributed_processing.async_result import gather
from distributed_processing.utils import fsclient

In [2]:
load_dotenv()
NS_PATH = getenv("NS_PATH")
MASTER_QUEUE = getenv("MASTER_QUEUE")
LAUNCH_SERVER = True

MASTER_QUEUE

'node_1'

In [3]:
# logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
# logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [4]:
if LAUNCH_SERVER:
    PYTHON_INTERPRETER = getenv("PYTHON_INTERPRETER")
    MULTI_WORKER_SCRIPT = getenv("MULTI_WORKER_SCRIPT")
    server_process = subprocess.Popen(
        [PYTHON_INTERPRETER, MULTI_WORKER_SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=False,
    )
    time.sleep(10)

In [5]:
# NBVAL_CHECK_OUTPUT
"""
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")
"""

client = fsclient(NS_PATH)
client.timeout = 20

Client with id: fs_client_1
Results queue: fs_client_1_responses


In [6]:
MASTER_QUEUE = client.to_requests_queue(MASTER_QUEUE)

In [7]:
client.update_registry_cache()

In [8]:
ns = fs_structs.structs.FSNamespace(NS_PATH)

In [9]:
# NBVAL_CHECK_OUTPUT
ns.registry.keys()

['method_queues_cleanup',
 'method_queues_create_worker',
 'method_queues_kill_all_processes',
 'method_queues_kill_process',
 'method_queues_kill_processes',
 'method_queues_list_processes',
 'nclients',
 'workers_queue_requests_node_1']

In [10]:
z1 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z3 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)

(z1, z2, z3)

(42080, 29772, 42012)

In [11]:
ps = client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)
ps

[(42080, 'worker1', 'fs_server_1'),
 (29772, 'worker1', 'fs_server_2'),
 (42012, 'worker1', 'fs_server_3')]

In [12]:
# NBVAL_CHECK_OUTPUT

len(ps)

3

In [13]:
# NBVAL_CHECK_OUTPUT

[(w, q) for _, w, q in ps]

[('worker1', 'fs_server_1'),
 ('worker1', 'fs_server_2'),
 ('worker1', 'fs_server_3')]

In [14]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync("kill_process", [z2], queue=MASTER_QUEUE)

True

In [15]:
client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)

[(42080, 'worker1', 'fs_server_1'), (42012, 'worker1', 'fs_server_3')]

In [16]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

2

In [ ]:
client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE)

In [ ]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

In [ ]:
z1 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z3 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)

In [ ]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

In [17]:
client.update_registry_cache()

In [18]:
y = client.rpc_async("add", [1, 0])

In [19]:
# NBVAL_CHECK_OUTPUT

y.get()

1

In [20]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

'method_queues_fake'


In [21]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"  # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


Error -32601 : The method does not exist/is not available.

 NoneType: None



In [22]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

'Esto es un error'

In [23]:
client.check_registry = "cache"

In [24]:
x = client.rpc_async("div", [1, 0])

In [25]:
try:
    x.get()
except Exception as e:
    print(e)

Error -3260 : Undefined error

 Traceback (most recent call last):
  File "\\ntcimdavinfo2\fondos\arquitectura_gestora\desarrollo\py_distributed_processing\distributed_processing\worker.py", line 362, in process_single_request
    result = func[request["method"]](*args, **kwargs)
  File "G:\arquitectura_gestora\desarrollo\py_distributed_processing\tests\fs_worker_multi.py", line 28, in div
    return x / y
           ~~^~~
ZeroDivisionError: division by zero



In [26]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print("Error")

Error


In [27]:
# x = client.rpc_sync("div", [1, 0])

In [28]:
# NBVAL_CHECK_OUTPUT


client.rpc_sync("add", [1, 1])

2

In [29]:
def f(x, y):
    return x + y


y = client.rpc_async_fn(f, [1, 2.0])

In [30]:
y.get()

3.0

In [31]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync_fn(f, [3.0, 2.0])

5.0

In [32]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

t = ("sleep", [10], {})
tp.append(t)
tp


[('mul', [0.9717317309464732, 0.6507851043245939], {}),
 ('mul', [0.46943929285445674, 0.5144601528562283], {}),
 ('lista', [0.9465663216741202, 0.959016200352219], {}),
 ('add', [0.7646080573776103, 0.2664946706761412], {}),
 ('lista', [0.6574958922618422, 0.5549743079977736], {}),
 ('add', [0.8197316876226767, 0.3909384138720322], {}),
 ('lista', [0.6024516794573556, 0.3633290805711983], {}),
 ('dic', [0.7112821606596426, 0.21848908745776174], {}),
 ('dic', [0.10347128810710193, 0.9147361035277288], {}),
 ('tupla', [0.9557268412221563, 0.9921985691484826], {}),
 ('dic', [0.7171903367434544, 0.517041621115311], {}),
 ('tupla', [0.7618899480330236, 0.1335576599426267], {}),
 ('dic', [0.19093247984676598, 0.08780742471003622], {}),
 ('div', [0.2913955548924434, 0.8515091318692524], {}),
 ('lista', [0.17124773710520924, 0.4202085463821993], {}),
 ('div', [0.40920883023090104, 0.7141751459104487], {}),
 ('mul', [0.5959672921958633, 0.8794624009559836], {}),
 ('lista', [0.536519185160797, 

In [33]:
tp = [
    ("div", [0.5273328219558507, 0.13835718442890943], {}),
    ("div", [0.7224042937278776, 0.6744424742714074], {}),
    ("mul", [0.16464192867054384, 0.2961055537758295], {}),
    ("lista", [0.24324779336838243, 0.4486736957376778], {}),
    ("dic", [0.222443603731179, 0.049201002693669005], {}),
    ("dic", [0.5892562777042863, 0.8190093828367946], {}),
    ("div", [0.9698052964451762, 0.2495544466990297], {}),
    ("add", [0.18281717945238662, 0.28456253134154685], {}),
    ("div", [0.8231058337900704, 0.4550312056693214], {}),
    ("mul", [0.6955981505190975, 0.2960636895338262], {}),
    ("add", [0.5793774647414703, 0.9353212122782703], {}),
    ("lista", [0.3893530442489298, 0.74231052966268], {}),
    ("div", [0.6419882052325951, 0.7661892480993476], {}),
    ("div", [0.049994104986677, 0.4562378471461709], {}),
    ("dic", [0.4734342157728231, 0.053714834674179035], {}),
    ("mul", [0.8092977961625194, 0.9195146847049076], {}),
    ("mul", [0.913778636227633, 0.6145608354175943], {}),
    ("lista", [0.7499808955353686, 0.5337360859450593], {}),
    ("dic", [0.6756209555432302, 0.9505296005797351], {}),
    ("dic", [0.5209295316393681, 0.9420323858687962], {}),
    ("div", [0.15809810362813237, 0.62619392590883], {}),
    ("dic", [0.29474022335742067, 0.893515494048087], {}),
    ("mul", [0.5349022233942071, 0.05757455715844495], {}),
    ("lista", [0.042102069906052586, 0.3469740971990074], {}),
    ("mul", [0.871021663889528, 0.007201521377221853], {}),
    ("lista", [0.6049724123450363, 0.26801461728942155], {}),
    ("dic", [0.8898534023208522, 0.5019296231298458], {}),
    ("lista", [0.9266421130454301, 0.9972178178045238], {}),
    ("mul", [0.48602513421859184, 0.199817263488683], {}),
    ("dic", [0.7163791721275343, 0.6950721561324937], {}),
    ("sleep", [10], {}),
]

In [34]:
fs = []
for t in tp:
    fs.append(client.rpc_async(t[0], t[1], t[2]))

In [35]:
# NBVAL_CHECK_OUTPUT

resultados = [f.get() for f in fs]
resultados

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [36]:
# NBVAL_CHECK_OUTPUT

fs = client.rpc_multi_async(tp)

In [37]:
time.sleep(3)

In [38]:
# NBVAL_CHECK_OUTPUT

# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory.
# If there are responses available in the queue or in the cache
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'PENDING']

In [39]:
r = client.wait_responses(timeout=2)
r

['fs_client_1:76']

In [40]:
# NBVAL_CHECK_OUTPUT

len(r)

1

In [41]:
time.sleep(7)

In [42]:
# NBVAL_CHECK_OUTPUT
# AsynResult.status updates information

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK']

In [43]:
# NBVAL_CHECK_OUTPUT

client.wait_responses()

[]

In [44]:
# NBVAL_CHECK_OUTPUT

try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

wait_responses: ['kk', 'qq'] neither in responses nor in pending.


In [45]:
client._update_cache_with_all_available_responses()

In [46]:
# NBVAL_CHECK_OUTPUT

client.pending

{}

In [47]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [48]:
fs = client.rpc_batch_async(tp)

In [49]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [50]:
# NBVAL_CHECK_OUTPUT

client.rpc_batch_sync(tp)

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [51]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

tp

[('mul', [0.024816943814542247, 0.5077750837089653], {}),
 ('fake', [0.5787501324127816, 0.6554459924758017], {}),
 ('mul', [0.5924830387775877, 0.6931602177789968], {}),
 ('add', [0.34844485662944935, 0.08555887969029197], {}),
 ('div', [0.4189195367953866, 0.13201832132706237], {}),
 ('fake', [0.47072910032271587, 0.7923030352963532], {}),
 ('div', [0.3593584597952595, 0.8646742810028946], {}),
 ('div', [0.047027543474433275, 0.7111547251311711], {}),
 ('div', [0.4370977317161804, 0.7766505793187793], {}),
 ('add', [0.2082912667756277, 0.6018786573626207], {}),
 ('mul', [0.43375745136570676, 0.7311588188489705], {}),
 ('mul', [0.4051029547810069, 0.4009320893079902], {}),
 ('fake', [0.8434499378108149, 0.22092000558085223], {}),
 ('div', [0.8253257154174483, 0.8102357575983701], {}),
 ('mul', [0.02822500847730558, 0.16400327022240113], {}),
 ('mul', [0.024188439457854316, 0.4842431546003496], {}),
 ('add', [0.7932608301514138, 0.2313448141091461], {}),
 ('add', [0.3683082720881774, 0

In [52]:
tp = [
    ("mul", [0.7355407026565478, 0.023519777553893007], {}),
    ("fake", [0.2520522260048764, 0.28054227055242864], {}),
    ("mul", [0.8838106264839363, 0.19639888038506714], {}),
    ("mul", [0.8857412900943293, 0.1253016179972587], {}),
    ("add", [0.0005226378683395039, 0.6568308199617323], {}),
    ("add", [0.15476536347494607, 0.4492869825171584], {}),
    ("fake", [0.7631067253940333, 0.019049359538004906], {}),
    ("fake", [0.4310102131915736, 0.675491507770126], {}),
    ("fake", [0.7140511506543913, 0.7837833237953286], {}),
    ("add", [0.0909525538014786, 0.44959184616881276], {}),
    ("add", [0.6627327150574825, 0.7401973301261011], {}),
    ("div", [0.21232540669537237, 0.7667374101737603], {}),
    ("add", [0.887254961441083, 0.21290364755712576], {}),
    ("fake", [0.7372649371990656, 0.3796617846297834], {}),
    ("add", [0.30864649241428155, 0.8777033159855755], {}),
    ("div", [0.4997997676145608, 0.45884184399530026], {}),
    ("fake", [0.0011733893324340494, 0.6016126834507816], {}),
    ("mul", [0.9307150630961242, 0.2943025202085412], {}),
    ("div", [0.6545197868355805, 0.11276464241684814], {}),
    ("mul", [0.6573680105483782, 0.13653818666573825], {}),
    ("add", [0.7959397704608381, 0.41576468218269147], {}),
    ("mul", [0.16347738503061415, 0.04898545483226813], {}),
    ("fake", [0.7815677830141265, 0.013945854620984632], {}),
    ("fake", [0.8020187012792446, 0.25875661742111045], {}),
    ("fake", [0.9043915109893543, 0.6876522434184933], {}),
    ("add", [0.929922781635924, 0.9540258004221696], {}),
    ("fake", [0.961238888823737, 0.4030318469598062], {}),
    ("div", [0.7466755564012741, 0.799944178434153], {}),
    ("mul", [0.15308290555092918, 0.45297088397324015], {}),
    ("mul", [0.3505532310800745, 0.5551384076337974], {}),
]

In [53]:
client.check_registry = "never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [54]:
# NBVAL_CHECK_OUTPUT

[f.safe_get() for f in fs]

[0.017299753708316164,
 None,
 0.17357941751386985,
 0.11098481677579876,
 0.6573534578300718,
 0.6040523459921044,
 None,
 None,
 None,
 0.5405443999702914,
 1.4029300451835836,
 0.27692063003323986,
 1.1001586089982087,
 None,
 1.1863498083998572,
 1.0892637063407844,
 None,
 0.2739117886652408,
 5.804299759281539,
 0.08975583613233945,
 1.2117044526435294,
 0.008008014060514455,
 None,
 None,
 None,
 1.8839485820580935,
 None,
 0.9334095759817276,
 0.06934209904859642,
 0.1946055624926752]

In [55]:
# NBVAL_CHECK_OUTPUT

client.responses

{}

In [56]:
client.check_registry = "cache"

In [57]:
# NBVAL_CHECK_OUTPUT

try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

Method fake does not exist/is not available.


In [58]:
client.check_registry = "never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [59]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [60]:
y = client.rpc_async("add", [1, 0])

In [61]:
# NBVAL_CHECK_OUTPUT

y.get(5)

1

In [62]:
# NBVAL_CHECK_OUTPUT


def f(x, y):
    return x + y


# "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn
client.check_registry = "never"
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [63]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

In [64]:
# NBVAL_CHECK_OUTPUT

y.get()

3.0

In [65]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

[]

In [66]:
client.clean_used()

In [67]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

[]

In [68]:
# client.rpc_sync_fn(print, ["hola"])

In [69]:
# NBVAL_CHECK_OUTPUT

tp = [
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
]
tp

[('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {})]

In [70]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

'requests_cola_1'

In [71]:
client.check_registry = "never"
fs = client.rpc_multi_async(tp, retry=True)

In [72]:
# NBVAL_CHECK_OUTPUT

gather(fs, 30, 5)

In [73]:
s = [f.status for f in fs]

In [74]:
# NBVAL_CHECK_OUTPUT

len(s), "OK" in s

(10, True)

In [75]:
[
    (time.time() if f.finished_time is None else f.finished_time) - f.creation_time
    for f in fs
]

[15.083526134490967,
 15.789429426193237,
 30.131815671920776,
 30.82742428779602,
 30.904051065444946,
 30.884284257888794,
 30.864866018295288,
 30.844022035598755,
 30.803096055984497,
 30.777788639068604]

In [76]:
[f.finished_time for f in fs]

[1774285698.0493784,
 1774285698.7748897,
 1774285713.138175,
 1774285713.857816,
 None,
 None,
 None,
 None,
 None,
 None]

In [77]:
client.pending

{'fs_client_1:177': 1774285683.0525563,
 'fs_client_1:178': 1774285683.0723264,
 'fs_client_1:179': 1774285683.0917463,
 'fs_client_1:180': 1774285683.1125903,
 'fs_client_1:181': 1774285683.1535127,
 'fs_client_1:182': 1774285683.178823}

In [78]:
fs[3].status

'OK'

In [79]:
fs[4].retry()

True

In [80]:
# [f.retry() for f in fs if not f.done()]
# no hace falta chequear si está pendiente.
[f.retry() for f in fs]

[False, False, False, False, True, True, True, True, True, True]

In [81]:
client.check_registry = "always"

In [82]:
# NBVAL_CHECK_OUTPUT

sorted(client.connector.all_queues_for_method("info"))

['requests_fs_server_1', 'requests_fs_server_3']

In [83]:
client.update_registry_cache()

In [84]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "Never"
client.all_queues_for_method("hola")

['requests_cola_1']

In [85]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"
a = client.rpc_async("info")

In [86]:
# client.rpc_sync("div", [2, 1], timeout=10)

In [87]:
x = a.get()
x

('fs_server_1',
 {'requests_fs_server_1': {'info'},
  'requests_cola_1': {'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'},
  'requests_cola_2': {'hola'},
  'requests_py_eval': {'eval_py_function'}})

In [88]:
# NBVAL_CHECK_OUTPUT

len(x), len(x[1])

(2, 4)

In [89]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_1"]

{'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'}

In [90]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_2"]

{'hola'}

In [91]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_py_eval"]

{'eval_py_function'}

In [92]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

'requests_cola_1'

In [93]:
client.set_default_queue("cola_1")

In [94]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

'requests_cola_1'

In [95]:
if LAUNCH_SERVER:
    client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE, timeout=120)
    if platform.system() == "Windows":
        subprocess.run(
            ["taskkill", "/F", "/T", "/PID", str(server_process.pid)],
            capture_output=True,
        )

    else:
        server_process.terminate()
        server_process.wait()

TimeoutError: 

In [ ]:
# NBVAL_CHECK_OUTPUT

ps = client.rpc_async("list_processes", [], queue=MASTER_QUEUE)
ps.safe_get(3)

In [ ]:
# NBVAL_CHECK_OUTPUT
try:
    y = client.rpc_async("add", [1, 0])
    y.safe_get(3)
except KeyError:
    print("Error")

Error
